# ijepa-core — Colab confirm-train (test gate 2/2)

Real-data, real-GPU confirmation that `pretrain/` actually trains: **ViT-S/16, 5 epochs**, on the
already-staged stratified DR subset. This is the second and final gate before `pretrain/` is
frozen (see `../CLAUDE.md`) — gate 1 is `../tests/smoke_test.py` (CPU, synthetic data, already
passing as of this repo's commit history).

**What this notebook does, in order:**
1. Mount Drive, clone/pull `ijepa-core` **into Drive** — persists across sessions, no re-cloning.
2. Locate the stratified subset already staged on Drive (built by the sibling `+ shaped JEPA`
   project's Phase-1 notebook) and copy/unzip it to **local Colab disk** — Drive-mounted FUSE reads
   are slow for thousands of small files; training reads locally instead.
3. Resolve a config: `model_name=vit_small`, `epochs=5`, auto-detected `use_bfloat16` (only if this
   session's GPU actually supports it — `main.py` hard-fails rather than silently degrading
   otherwise, so guessing wrong here would just fail loudly, not corrupt anything).
4. Run `pretrain/main.py` for real. Outputs (resolved config, checkpoint, CSV log) land under a
   **Drive-persistent** run folder, not ephemeral `/content`.
5. Plot the loss curve, assert it's finite and trending down, verify a checkpoint-resume round trip
   on real data.
6. Print the exact `CLAUDE.md` Status-section update to paste in once this passes — freezing
   `pretrain/` is a manual step (commit + push), this notebook only produces the evidence.

Runtime: **GPU** (Runtime → Change runtime type → GPU) before running.


## 1. Mount Drive, clone/pull ijepa-core

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
IJEPA_CORE_DIR = DRIVE_ROOT / "ijepa-core"
IJEPA_CORE_REMOTE = "https://github.com/khalilLaatiris/ijepa-core.git"

if (IJEPA_CORE_DIR / ".git").exists():
    print(f"ijepa-core already cloned at {IJEPA_CORE_DIR} -- pulling latest")
    subprocess.run(["git", "-C", str(IJEPA_CORE_DIR), "pull", "--ff-only"], check=True)
else:
    print(f"cloning ijepa-core -> {IJEPA_CORE_DIR}")
    subprocess.run(["git", "clone", IJEPA_CORE_REMOTE, str(IJEPA_CORE_DIR)], check=True)

print("HEAD:", subprocess.run(["git", "-C", str(IJEPA_CORE_DIR), "log", "--oneline", "-1"],
                               capture_output=True, text=True, check=True).stdout.strip())


## 2. Locate + stage the stratified subset (download cell)

Reads from the Phase-1 Colab notebook's known output location
(`+ shaped JEPA/notebooks/phase1_stratified_subset_colab.ipynb`): grade-classed `ImageFolder`s (or
their packaged `.zip`s) under `MyDrive/retina_phase1/subsets/`. This notebook does **not** rebuild
the subset — if none exists yet, run that notebook first.


In [ ]:
import shutil
import zipfile

SUBSET_ROOT = DRIVE_ROOT / "retina_phase1" / "subsets"
#@markdown Leave blank to auto-detect (only works if exactly one `subset_*` exists on Drive).
SUBSET_NAME = ""  #@param {type:"string"}

assert SUBSET_ROOT.exists(), (
    f"{SUBSET_ROOT} does not exist -- run the Phase-1 stratified-subset notebook first "
    f"('+ shaped JEPA/notebooks/phase1_stratified_subset_colab.ipynb')."
)

if SUBSET_NAME:
    candidates = [SUBSET_ROOT / SUBSET_NAME]
else:
    candidates = sorted(p for p in SUBSET_ROOT.glob("subset_*") if p.is_dir() or p.suffix == ".zip")
    # prefer a directory over its own .zip if both exist for the same base name
    seen_stems = set()
    deduped = []
    for p in candidates:
        stem = p.stem if p.suffix == ".zip" else p.name
        if stem in seen_stems:
            continue
        seen_stems.add(stem)
        deduped.append(p)
    candidates = deduped

assert candidates, f"no subset_* found under {SUBSET_ROOT} -- run Phase 1 first."
assert len(candidates) == 1, (
    f"multiple subsets found: {[c.name for c in candidates]} -- set SUBSET_NAME explicitly above."
)
SUBSET_SRC = candidates[0]
print(f"using subset: {SUBSET_SRC}")

LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_SUBSET_DIR = LOCAL_DATA_ROOT / "subset_phase1"

if LOCAL_SUBSET_DIR.exists() and any(LOCAL_SUBSET_DIR.iterdir()):
    print(f"already staged locally at {LOCAL_SUBSET_DIR}, skipping copy/unzip")
else:
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if SUBSET_SRC.suffix == ".zip":
        print(f"unzipping {SUBSET_SRC} -> {LOCAL_SUBSET_DIR} ...")
        LOCAL_SUBSET_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SUBSET_SRC) as z:
            z.extractall(LOCAL_SUBSET_DIR)
    else:
        print(f"copying {SUBSET_SRC} -> {LOCAL_SUBSET_DIR} (this reads from Drive once, then trains locally) ...")
        shutil.copytree(SUBSET_SRC, LOCAL_SUBSET_DIR)

n_grade_dirs = sum(1 for p in LOCAL_SUBSET_DIR.iterdir() if p.is_dir())
n_images = sum(1 for _ in LOCAL_SUBSET_DIR.rglob("*.jpeg")) + sum(1 for _ in LOCAL_SUBSET_DIR.rglob("*.jpg"))
assert n_grade_dirs > 0, f"{LOCAL_SUBSET_DIR} has no grade subdirectories after staging."
print(f"staged: {n_grade_dirs} grade dirs, {n_images} images at {LOCAL_SUBSET_DIR}")

# pretrain/'s vendored ImageNet loader hardcodes <root_path>/<image_folder>/train/ -- symlink the
# level in, same fix used by every HPC-MARWAN job for this dataset.
IJEPA_DATA_ROOT = LOCAL_DATA_ROOT / "ijepa_view"
(IJEPA_DATA_ROOT / "subset_phase1").mkdir(parents=True, exist_ok=True)
train_link = IJEPA_DATA_ROOT / "subset_phase1" / "train"
if train_link.is_symlink() or train_link.exists():
    train_link.unlink()
train_link.symlink_to(LOCAL_SUBSET_DIR, target_is_directory=True)
print(f"train/ view -> {train_link} -> {LOCAL_SUBSET_DIR}")


## 3. Resolve config: vit_small, 5 epochs, auto bf16

In [ ]:
import sys
import torch
import yaml

sys.path.insert(0, str(IJEPA_CORE_DIR / "pretrain"))

RUN_ROOT = DRIVE_ROOT / "ijepa-core-runs" / "confirm_train"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RUN_ROOT / "ckpts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "no GPU visible -- Runtime -> Change runtime type -> GPU, then re-run."
gpu_name = torch.cuda.get_device_name(0)
use_bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name} | bf16 supported: {use_bf16}")

with open(IJEPA_CORE_DIR / "pretrain" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["meta"]["model_name"] = "vit_small"
cfg["meta"]["use_bfloat16"] = bool(use_bf16)
cfg["data"]["root_path"] = str(IJEPA_DATA_ROOT)
cfg["data"]["image_folder"] = "subset_phase1"
cfg["data"]["num_workers"] = 2
cfg["logging"]["folder"] = str(CKPT_DIR) + "/"
cfg["logging"]["write_tag"] = "confirm"
cfg["optimization"]["epochs"] = 5

latest = CKPT_DIR / f"{cfg['logging']['write_tag']}-latest.pth.tar"
cfg["meta"]["load_checkpoint"] = latest.exists()

RESOLVED_CONFIG = RUN_ROOT / "config_resolved.yaml"
with open(RESOLVED_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f)

print(f"resolved config -> {RESOLVED_CONFIG}")
print(f"  model_name={cfg['meta']['model_name']}  use_bfloat16={cfg['meta']['use_bfloat16']}")
print(f"  root_path={cfg['data']['root_path']}  image_folder={cfg['data']['image_folder']}")
print(f"  epochs={cfg['optimization']['epochs']}  batch_size={cfg['data']['batch_size']}")
print(f"  logging.folder={cfg['logging']['folder']}")
print(f"  load_checkpoint={cfg['meta']['load_checkpoint']} (auto-resume if a prior run left a checkpoint)")


## 4. Train (real run — this is the actual gate)

In [ ]:
import os
os.chdir(IJEPA_CORE_DIR / "pretrain")
from src.train import main as train_main
train_main(args=cfg, device=torch.device("cuda:0"))


## 5. Verify: loss curve, finite + decreasing, checkpoint present

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = CKPT_DIR / f"{cfg['logging']['write_tag']}_r0.csv"
log_df = pd.read_csv(csv_path, header=None,
                      names=["epoch", "itr", "loss", "mask_a", "mask_b", "time_ms"])

assert log_df["loss"].notna().all(), "NaN loss encountered -- training is broken, do not freeze pretrain/."
assert (log_df["loss"] < float("inf")).all(), "Inf loss encountered -- training is broken."

first_epoch_avg = log_df[log_df["epoch"] == 1]["loss"].mean()
last_epoch_avg = log_df[log_df["epoch"] == log_df["epoch"].max()]["loss"].mean()
print(f"epoch 1 avg loss: {first_epoch_avg:.4f}")
print(f"epoch {log_df['epoch'].max()} avg loss: {last_epoch_avg:.4f}")
assert last_epoch_avg < first_epoch_avg, (
    f"loss did not decrease ({first_epoch_avg:.4f} -> {last_epoch_avg:.4f}) -- "
    f"investigate before freezing pretrain/, this is the whole point of the gate."
)

plt.figure(figsize=(8, 4))
plt.plot(log_df.index, log_df["loss"])
plt.xlabel("iteration"); plt.ylabel("loss"); plt.title("ijepa-core confirm-train: 5 epochs, vit_small")
plt.tight_layout()
fig_path = RUN_ROOT / "loss_curve.png"
plt.savefig(fig_path)
plt.show()
print(f"saved loss curve -> {fig_path}")

latest_ckpt = CKPT_DIR / f"{cfg['logging']['write_tag']}-latest.pth.tar"
assert latest_ckpt.exists(), f"expected checkpoint at {latest_ckpt}, not found."
print(f"checkpoint OK: {latest_ckpt}  ({latest_ckpt.stat().st_size / 1e6:.1f} MB)")


## 6. Checkpoint-resume round trip (real data)

Bumps `epochs` to 6 with `load_checkpoint: true` and confirms training resumes from epoch 5 rather
than restarting — mirrors the CPU smoke test's CSV-row-count check, adapted for a real run (a real
run's CSV already has enough rows that a coarse row-count check is a weaker signal here; instead
this checks the resumed run's first logged epoch number directly).


In [ ]:
cfg["optimization"]["epochs"] = 6
cfg["meta"]["load_checkpoint"] = True
with open(RESOLVED_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f)

train_main(args=cfg, device=torch.device("cuda:0"))

log_df2 = pd.read_csv(csv_path, header=None,
                       names=["epoch", "itr", "loss", "mask_a", "mask_b", "time_ms"])
epochs_seen = sorted(log_df2["epoch"].unique())
print(f"epochs present in log after resume: {epochs_seen}")
assert epochs_seen == list(range(1, 7)), (
    f"expected epochs 1..6 with no repeats (proof epoch 5 was NOT retrained from scratch), "
    f"got {epochs_seen} -- resume may have silently failed."
)
print("RESUME VERIFIED: epoch 6 trained once, on top of the epoch-5 checkpoint, no retraining.")


## Done — if all assertions above passed

Both test-gate stages have now passed:
1. `tests/smoke_test.py` (CPU, synthetic data) — see this repo's commit history.
2. This notebook (GPU, real data, 5+1 epochs, vit_small) — passed just now.

`pretrain/` can now be marked frozen. **This is a manual step** (this notebook has no push
credentials and shouldn't be given any): update `ijepa-core/CLAUDE.md`'s `## Status` section on
your own machine with the block below (fill in the GPU name and losses printed above), then commit
and push.


In [ ]:
print(f'''
- Colab confirm run: PASSED. GPU={gpu_name}, model=vit_small, 5+1 epochs (fresh + resumed).
  Epoch 1 avg loss: {first_epoch_avg:.4f}. Epoch 5 avg loss: {last_epoch_avg:.4f}.
  Resume verified: epochs 1..6 present exactly once each, no retraining from scratch.
- pretrain/ is now FROZEN per both gates passing (smoke_test.py + this notebook).
  No silent edits from here on -- see the freeze rule above.
''')
